# Pydantic Data Validation Example

In [1]:
from pydantic import BaseModel

# Validate the User Data
class User(BaseModel):
    id: int
    name: str


usr = User(id=1, name="Rishabh")
print(usr)

id=1 name='Rishabh'


In [2]:
# Use Default Values
class User(BaseModel):
    id: int
    name: str
    signup_ts: str | None = None
    salary: float | None = None

usr1 = User(id="2", name="Peter", salary=50000)
print(usr1)

id=2 name='Peter' signup_ts=None salary=50000.0


In [4]:
from dataclasses import dataclass

@dataclass
class User():
    id: int
    name: str
    signup_ts: str | None = None
    salary: float | None = None

usr2 = User(id="2", name="Peter", salary=50000)
print(usr2)

User(id='2', name='Peter', signup_ts=None, salary=50000)


## Scenario: User Registration API Input


In [5]:
# You're building a backend service that receives JSON payloads from a front-end form:

{
  "username": "johndoe",
  "password": "secret123",
  "email": "john@example.com",
  "age": "30",
  "join_date": "2025-01-01"
}

{'username': 'johndoe',
 'password': 'secret123',
 'email': 'john@example.com',
 'age': '30',
 'join_date': '2025-01-01'}

## You need to:
- Validate the email format

- Ensure age is a positive integer

- Ensure password meets minimum length requirements

- Convert "30" to integer

- Parse "2025-01-01" into a datetime.date

In [7]:
from pydantic import BaseModel, EmailStr, field_validator, ValidationError
from datetime import date

class UserRegistration(BaseModel):
    username: str
    password: str
    email: EmailStr
    age: int
    join_date: date

    @field_validator("age", mode='after')
    @classmethod
    def validate_age(cls, value):
        if value < 0:
            raise ValueError("Age must be positive")
        return value
    
    @field_validator("password",  mode='after')
    @classmethod
    def validate_age(cls, value):
        if len(value) < 10:
            raise ValueError("Password must be at least 10 characters long")
        return value

/var/folders/gs/pjt6j0rs4pv374n5d5vx5wrm0000gn/T/ipykernel_59065/1542669406.py:20: UserWarning: `validate_age` overrides an existing Pydantic `@field_validator` decorator
  def validate_age(cls, value):


In [9]:
# Simulate incoming JSON
incoming_data = {
    "username": "johndoe",
    "password": "secret123456789",
    "email": "john@example.com",
    "age": "30",              
    "join_date": "2023-01-01" 
}

try:
    user = UserRegistration(**incoming_data)
    #print(user)
    print("JSON output:", user.model_dump_json())
except ValidationError as e:
    print("❌ Validation Error:")
    print(e)


JSON output: {"username":"johndoe","password":"secret123456789","email":"john@example.com","age":30,"join_date":"2023-01-01"}


## Nested Models

In [10]:
class Address(BaseModel):
    street: str
    city: str
    zip_code: str

class UserRegistrationWithAddress(UserRegistration):
    address: Address

incoming_data = {
    "username": "johndoe",
    "password": "secret123534323",
    "email": "john@example.com",
    "age": "30",
    "join_date": "2025-01-01",
    "address": {
        "street": "123 Main St",
        "city": "New York",
        "zip_code": "10001"
    }
}

try:
    userWithAdd = UserRegistrationWithAddress(**incoming_data)
    #print(user)
    print("JSON output:", userWithAdd.model_dump_json())
except ValidationError as e:
    print("❌ Validation Error:")
    print(e)

JSON output: {"username":"johndoe","password":"secret123534323","email":"john@example.com","age":30,"join_date":"2025-01-01","address":{"street":"123 Main St","city":"New York","zip_code":"10001"}}
